# Consolidated sweep - ingest many tables in ONE job task

One `lakefed_ingest_mt_sweep` run processes a whole **task_collection** of (tenant db, table)
tasks in this single notebook, using a thread pool + Spark **FAIR scheduler pools** so each
table's work overlaps on the cluster. This decouples throughput from the workspace job limits
(2,000 concurrent task runs; 10,000 run-submits/hour) that a task-per-table design hits - the
throttle becomes compute (cluster cores), which is the intended cost model. Shard by
`task_collection` for horizontal scale: run one sweep per collection.

Notes:
- Cluster must run the **FAIR** scheduler (`spark.scheduler.mode=FAIR`) for the pools to apply.
- All Change-Tracking / source-touching work is in **ct_engine.py** (Alex, workstreams A/B/F) -
  this notebook is pure orchestration + Lakebase checkpointing.
- Per-table failures are isolated (one bad table does not fail the collection) and written to
  the Lakebase checkpoint `status` for triage.

In [ ]:
# Cluster task. Bootstrap: put this notebook's dir on sys.path and import the shared modules.
import sys, os, json

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)

import state_store as ss
from state_store import StateStore
import ct_engine
import parallel


In [ ]:
# Params + connect to Lakebase (SDK-minted OAuth token; no stored password).
def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)


task_collection = widget("task_collection", "example_cohort")
db_instance = widget("db_instance", "lakefed-ingest-mt")
db_name = widget("db_name", "lakefed_ingest_mt")
parallelism = int(widget("parallelism", "16"))

store = StateStore.connect_lakebase(db_instance, db_name)
configs = store.list_task_configs(task_collection)
print(f"collection {task_collection}: {len(configs)} tables to sweep (parallelism={parallelism})")

In [ ]:
# Per-(tenant db, table) state machine, run in-process. The CT/source ops are ct_engine
# placeholders (Alex, A/B/F); everything here is orchestration + Lakebase checkpointing.
def ingest_one(cfg):
    db, schema, table = cfg["src_database"], cfg["src_schema"], cfg["src_table"]
    sink = f"{cfg['sink_catalog']}.{cfg['sink_schema']}.{cfg['sink_table']}"
    ct_engine.ensure_sink_table(spark, cfg, sink)
    cp = store.get_checkpoint(db, schema, table)

    # seeding -> seed; cdc -> reseed (retention lapsed) or increment.
    if ss.decide_action(cp["phase"], cp["ct_version"], None) == "seed":
        v0 = ct_engine.current_version(cfg)          # capture BEFORE the seed
        ct_engine.seed(spark, cfg, sink)
        store.advance_version(db, schema, table, ct_version=v0, phase=ss.CDC, detail="seed complete")
        return "seeded"

    mv = ct_engine.min_valid_version(cfg)
    if ss.decide_action(cp["phase"], cp["ct_version"], mv) == "reseed":
        store.enqueue_reseed(db, schema, table, reason="retention_miss")  # flips phase -> seeding
        return "reseed_enqueued"

    new_version = ct_engine.current_version(cfg)     # upper bound BEFORE the read
    ct_engine.read_and_merge_ct(spark, cfg, sink, cp["ct_version"])
    store.advance_version(db, schema, table, ct_version=new_version)  # only AFTER the MERGE commits
    return "incremented"


def worker(cfg):
    try:
        return ingest_one(cfg)
    except Exception as e:
        # Record the failure in Lakebase for triage, then re-raise so run_parallel isolates it.
        try:
            store.set_status(cfg["src_database"], cfg["src_schema"], cfg["src_table"], "failed", str(e)[:500])
        except Exception:
            pass
        raise


def set_pool(pool):
    # Each table's Spark jobs (e.g. its MERGE) run in their own FAIR pool so they overlap fairly.
    spark.sparkContext.setLocalProperty("spark.scheduler.pool", pool)


In [ ]:
# Sweep the collection: one thread per concurrency slot, round-robin FAIR pools, failures isolated.
results = parallel.run_parallel(
    configs, worker, parallelism, set_pool=set_pool, item_id=lambda c: c["id"]
)
summary = parallel.summarize(results)
summary["task_collection"] = task_collection
print(json.dumps(summary, indent=2))

failures = [r for r in results if r["status"] == "failed"]
if failures:
    print("failures:", json.dumps(failures, default=str)[:2000])

dbutils.notebook.exit(json.dumps(summary, default=str))